<h1>Chapter 8 - Agentic RAG</h1>
<i>Building a multi-agent system using OpenAI SDK.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch08_agentic_rag/8.6_sales_negotiation_agent_openai_sdk/building_agents_with_openai_sdk.ipynb)

---

This notebook is for Chapter 8 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


## Building a multi-agent system using OpenAI SDK

### Install Required Packages

In [1]:
%pip install openai-agents==0.9.2 chromadb==1.5.0
%pip install chromadb==1.5.0
%pip install openai-agents==0.9.2

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


### Load secrets

If you run this code in Google Colab, save your OpenAI API key in the secrets and access it by

In [2]:
import os
import sys
from dotenv import load_dotenv

# Check if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    try:
        from google.colab import userdata  # type: ignore

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except ModuleNotFoundError:
        pass
else:
    load_dotenv()

### Load sample files

This notebook uses sample Word and PDF files.

When running the notebook on Google Colab, uncomment the code below to download the `datasets` directory from the Github repo.

In [9]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone --no-checkout https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git
    %cd RAG-with-Python-Cookbook
    !git sparse-checkout init --cone
    !git sparse-checkout set datasets
    !git checkout
    !cp -r datasets /content/datasets
    print("\u2713 Datasets downloaded to /content/datasets")

    # set the datasets_dir
    datasets_dir = "./datasets"
else:
    datasets_dir = "../../datasets"
    print("\u26a0 Running locally. Using ../datasets/ directory")

⚠ Running locally. Using ../datasets/ directory


## Code

In [10]:
import os
import shutil
import chromadb
from chromadb.config import Settings

def fill_chroma_db():
    chroma_dir = "./chroma_email_history_db"
    # If the directory exists, remove it to overwrite
    if os.path.exists(chroma_dir):
        shutil.rmtree(chroma_dir)
    client = chromadb.Client(Settings(persist_directory=chroma_dir))
    collection = client.get_or_create_collection("email_history")

    with open(f'{datasets_dir}/sample_data_negotiation_agents/sample_email_history.txt', "r", encoding="utf-8") as f:
        email_text = f.read()

    email_docs = [
        e.strip() for e in email_text.split("\n---\n") if e.strip()
    ]

    for idx, doc in enumerate(email_docs):
        collection.add(documents=[doc], ids=[f"email_{idx+1}"])
        print(f"Added email_{idx+1}")

fill_chroma_db()


2026-02-24 21:34:37.064846774 [W:onnxruntime:Default, device_discovery.cc:131 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename ""5620e0c7-8062-4dce-aeb7-520c7ef76171"" dit not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


Added email_1


In [11]:
import chromadb
from chromadb.config import Settings
from agents import function_tool

@function_tool
def query_database(query_text, n_results=3):
    """
    Query emails using semantic search from a local ChromaDB collection.
    """

    # Set up ChromaDB client and collection
    chroma_dir = "./chroma_email_history_db"
    client = chromadb.Client(Settings(persist_directory=chroma_dir))
    collection = client.get_or_create_collection("email_history")

    # Perform the query
    results = collection.query(
        query_texts=[query_text],
        n_results=n_results,
        include=["documents"],
    )
    return {
        "query": query_text,
        "documents": (
            results["documents"][0] if results["documents"] else []
        ),
    }

In [12]:
from agents import Agent, Runner
from agents import trace

# Create a negotiation agent with email search capability
salesman_agent = Agent(
    name="Negotiation Agent",
    instructions="""You are a skilled negotiation agent
    representing a salesperson.

    Your role:
    - Analyze the current email conversation history
    - Search for relevant information using the query_database tool
    - Generate informed responses with strategic counter-offers

    Process:
    1. Use query_database to find relevant context from email history
    2. Analyze customer concerns and pricing constraints
    3. Craft a professional response with a competitive counter-offer
    4. Maintain a collaborative tone while protecting profit margins
    """,
    tools=[query_database],
    model="gpt-5"
)

# Helper function to run the negotiation agent
async def negotiate_with_customer(email_history):
    with trace("Negotiation Response"):
        result = await Runner.run(
            salesman_agent,
            f"Email conversation history:\n{email_history}"
        )
    return result.final_output

In [13]:
from agents import Agent, Runner

async def run_negotiation_process():
    """
    Orchestrates a negotiation between customer and salesperson agents
    using a moderator agent to facilitate the conversation.
    """
    # Load the email history from file
    with open(f'{datasets_dir}/sample_data_negotiation_agents/sample_email_history.txt', "r", encoding="utf-8") as f:
        email_history = f.read()

    # Define the customer agent
    customer_agent = Agent(
        name="Customer",
        instructions="""Negotiate for the best laptop deal.
                       Be polite but persistent.
                       Respond to the email chain below.""",
        model="gpt-5-mini",
    )

    # Convert agents to tools for the moderator
    customer_agent_tool = customer_agent.as_tool(
        tool_name="customer",
        tool_description=(
            "Tool that represents the customer in negotiations."
        )
    )

    salesman_agent_tool = salesman_agent.as_tool(
        tool_name="salesman",
        tool_description=(
            "Tool that represents the salesperson in negotiations."
        )
    )

    # Define the moderator agent with clear instructions
    moderator_instructions = """
    You are a moderator facilitating negotiation between
    customer and salesperson.

    Process:
    1. Receive email history with the last message from customer
    2. Use salesman_agent_tool to generate salesperson response
    3. Append response to email history
    4. Use customer_agent_tool to generate customer response
    5. Continue alternating until agreement or breakdown

    Rules:
    - Aim for mutually beneficial agreement
    """

    moderator_agent = Agent(
        name="Moderator",
        instructions=moderator_instructions,
        model="gpt-5-mini",
        tools=[customer_agent_tool, salesman_agent_tool]
    )

    # Execute the negotiation with tracing
    with trace("Negotiation Process"):
        responses = await Runner.run(
            moderator_agent,
            f"Begin negotiation with this email history: {email_history}"
        )

    return responses


In [14]:
import os
import shutil
import chromadb
from chromadb.config import Settings

def fill_chroma_db():
    chroma_dir = "./chroma_email_history_db"
    # If the directory exists, remove it to overwrite
    if os.path.exists(chroma_dir):
        shutil.rmtree(chroma_dir)
    client = chromadb.Client(Settings(persist_directory=chroma_dir))
    collection = client.get_or_create_collection("email_history")

    with open(f"{datasets_dir}/sample_data_negotiation_agents/sample_email_history.txt", "r", encoding="utf-8") as f:
        email_text = f.read()

    email_docs = [e.strip() for e in email_text.split("\n---\n") if e.strip()]

    for idx, doc in enumerate(email_docs):
        collection.add(documents=[doc], ids=[f"email_{idx+1}"])
        print(f"Added email_{idx+1}")

fill_chroma_db()

Added email_1


In [15]:
import chromadb
from chromadb.config import Settings

@function_tool
def query_database(query_text: str, n_results: int = 3) -> dict:
    """
    Query emails using semantic search from a local ChromaDB collection.
    """

    # Set up ChromaDB client and collection
    chroma_dir = "./chroma_email_history_db"
    client = chromadb.Client(Settings(persist_directory=chroma_dir))
    collection = client.get_or_create_collection("email_history")

    # Perform the query
    results = collection.query(
        query_texts=[query_text],
        n_results=n_results,
        include=["documents"],
    )
    return {
        "query": query_text,
        "documents": results["documents"][0] if results["documents"] else [],
    }

In [16]:
from agents import Agent, Runner
from agents import trace

# Create a negotiation agent with email search capability
salesman_agent = Agent(
    name="Negotiation Agent",
    instructions="""You are a skilled negotiation agent representing a salesperson.

    Your role:
    - Analyze the current email conversation history
    - Search for relevant information using the query_database tool
    - Generate informed responses with strategic counter-offers

    Process:
    1. Use query_database to find relevant context from email history
    2. Analyze customer concerns and pricing constraints
    3. Craft a professional response with a competitive counter-offer
    4. Maintain a collaborative tone while protecting profit margins
    """,
    tools=[query_database],
    model="gpt-5"
)

# Helper function to run the negotiation agent
async def negotiate_with_customer(email_history: str) -> str:
    with trace("Negotiation Response"):
        result = await Runner.run(
            salesman_agent,
            f"Email conversation history:\n{email_history}"
        )
    return result.final_output

In [17]:
from agents import trace

async def demonstrate_negotiation():
    with open(f"{datasets_dir}/sample_data_negotiation_agents/sample_email_history.txt",
              "r", encoding="utf-8") as f:
        email_history = f.read()

    # Generate negotiation response using the agent
    with trace("Automated Customer Response"):
        response = await negotiate_with_customer(email_history)

    print(response)
    return response

await demonstrate_negotiation()

Trace already exists. Creating a new trace, but this is probably a mistake.


I wasn’t able to retrieve the email thread (database returned “404: Not Found”). If you can share the last inbound email and a few details (budget cap, current quote/term, seat count, renewal date, any competitor mentioned), I’ll tailor this precisely. In the meantime, here’s a ready-to-send response with strategic counter-offers you can tweak.

Subject: Let’s align on a plan that fits your budget and timeline

Hi [First Name],

Thanks for the candid feedback on pricing. I want to make sure we protect the outcomes we discussed—[key outcomes/use cases]—while keeping this within your budget and timeline. Based on what you shared, here are two paths that balance cost and value:

Option A — Lean and simple (1-year):
- Price: $[X]/yr with a 7–8% concession
- Onboarding: 50% off implementation (or swap for [2] training sessions)
- Terms: Net 30, with a 12-month price lock

Option B — Best total value (2-year):
- Price: $[X]/yr with a 12% concession (annual prepay)
- Adds: +10% additional sea

'I wasn’t able to retrieve the email thread (database returned “404: Not Found”). If you can share the last inbound email and a few details (budget cap, current quote/term, seat count, renewal date, any competitor mentioned), I’ll tailor this precisely. In the meantime, here’s a ready-to-send response with strategic counter-offers you can tweak.\n\nSubject: Let’s align on a plan that fits your budget and timeline\n\nHi [First Name],\n\nThanks for the candid feedback on pricing. I want to make sure we protect the outcomes we discussed—[key outcomes/use cases]—while keeping this within your budget and timeline. Based on what you shared, here are two paths that balance cost and value:\n\nOption A — Lean and simple (1-year):\n- Price: $[X]/yr with a 7–8% concession\n- Onboarding: 50% off implementation (or swap for [2] training sessions)\n- Terms: Net 30, with a 12-month price lock\n\nOption B — Best total value (2-year):\n- Price: $[X]/yr with a 12% concession (annual prepay)\n- Adds: +10

In [18]:
with open(f"{datasets_dir}/sample_data_negotiation_agents/sample_email_history.txt", "r", encoding="utf-8") as f:
    email_history = f.read()

email_history

'404: Not Found'

In [19]:
from agents import Agent, Runner

async def run_negotiation_process():
    """
    Orchestrates a negotiation between customer and salesperson agents
    using a moderator agent to facilitate the conversation.
    """
    # Load the email history from file
    with open(f"{datasets_dir}/sample_data_negotiation_agents/sample_email_history.txt", "r", encoding="utf-8") as f:
        email_history = f.read()

    # Define the customer agent
    customer_agent = Agent(
        name="Customer",
        instructions="""Negotiate for the best laptop deal.
                       Be polite but persistent.
                       Respond to the email chain below.""",
        model="gpt-5-mini",
    )

    # Convert agents to tools for the moderator
    customer_agent_tool = customer_agent.as_tool(
        tool_name="customer",
        tool_description="Tool that represents the customer in negotiations."
    )

    salesman_agent_tool = salesman_agent.as_tool(
        tool_name="salesman",
        tool_description="Tool that represents the salesperson in negotiations."
    )

    # Define the moderator agent with clear instructions
    moderator_instructions = """
    You are a moderator facilitating negotiation between customer and salesperson.

    Process:
    1. Receive email history with the last message from customer
    2. Use salesman_agent_tool to generate salesperson response
    3. Append response to email history
    4. Use customer_agent_tool to generate customer response
    5. Continue alternating until agreement or breakdown

    Rules:
    - Aim for mutually beneficial agreement
    """

    moderator_agent = Agent(
        name="Moderator",
        instructions=moderator_instructions,
        model="gpt-5-mini",
        tools=[customer_agent_tool, salesman_agent_tool]
    )

    # Execute the negotiation with tracing
    with trace("Negotiation Process"):
        responses = await Runner.run(
            moderator_agent,
            f"Begin negotiation with this email history: {email_history}"
        )

    return responses

In [20]:
responses = await run_negotiation_process()

In [21]:
from agents import Agent, Runner

async def run_negotiation_process_with_handoffs():
    """
    Direct negotiation between customer and salesperson agents
    using handoffs to alternate between them.
    """
    # Load the email history from file
    with open(f"{datasets_dir}/sample_data_negotiation_agents/sample_email_history.txt", "r", encoding="utf-8") as f:
        email_history = f.read()

    # Define the salesperson agent
    salesman_agent = Agent(
        name="Salesperson",
        instructions="""You are a laptop salesperson negotiating a deal.
                       Make reasonable offers and try to close the sale.
                       If you reach agreement, clearly state 'DEAL AGREED'.
                       If negotiation isn't working after 3 rounds, politely end it.""",
        model="gpt-5-mini",
        handoff_description="Hand off to customer for their response to your offer."
    )

    # Define the customer agent with handoff capabilities
    customer_agent = Agent(
        name="Customer",
        instructions="""Negotiate for the best laptop deal.
                       Be polite but persistent.
                       Make counteroffers or accept good deals.
                       If you accept a deal, clearly state 'DEAL ACCEPTED'.
                       If you can't reach agreement after 3 rounds, politely walk away.""",
        model="gpt-5-mini",
        handoff_description="Hand off to salesperson for their response to your counteroffer."
    )

    # Convert each agent to a tool for the other
    customer_tool = customer_agent.as_tool(
        tool_name="handoff_to_customer",
        tool_description="Hand the conversation to the customer agent."
    )

    salesman_tool = salesman_agent.as_tool(
        tool_name="handoff_to_salesperson",
        tool_description="Hand the conversation to the salesperson agent."
    )

    # Give each agent the ability to hand off to the other
    customer_agent.tools = [salesman_tool]
    salesman_agent.tools = [customer_tool]

    # Start with the salesperson responding to the email history
    with trace("Direct Negotiation Process"):
        responses = await Runner.run(
            salesman_agent,
            f"Respond to this email history and begin negotiation: {email_history}"
        )

    return responses

In [22]:
responses = await run_negotiation_process_with_handoffs()